# Bài 30 — chuỗi luyện Easy · LeetCode #145 — Binary Tree Postorder Traversal

**Mức độ:** Easy · **Ngôn ngữ:** Python 3 · **Đề gốc:** https://leetcode.com/problems/binary-tree-postorder-traversal/

**Mục tiêu học tập**

- Đọc đề và tự phân biệt input/output, constraints và ví dụ biên.
- Nắm cú pháp Python theo từng cell thực nghiệm trước khi ghép thuật toán.
- Hiểu brute force/baseline trước, sau đó giải thích mấu chốt tối ưu bằng bất biến.
- Trace từng bước với dữ liệu dễ sai, không xem code như một hộp đen.
- Chạy lời giải chuẩn LeetCode, test mẫu, edge case, randomized/exhaustive khi phù hợp.
- Tự giải **6 bài luyện từ dễ đến khó**, chỉ xem gợi ý/đáp án cuối sau khi thử.

**Cách học:** đọc Markdown → tự dự đoán output → chạy cell Python ngay sau đó → giải thích lại bằng lời. Các cell TODO không chạy tự động và chỉ phục vụ tự luyện.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Trả danh sách giá trị khi duyệt cây theo thứ tự **Left → Right → Root**. Chỉ ghi node **sau khi** đã xử lý cả hai cây con.

### Ví dụ quan trọng

`[1,None,2,3] → [3,2,1]`; `[4,2,6,1,3,5,7] → [1,3,2,5,7,6,4]`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. TreeNode là đối tượng, không phải phần tử của mảng

`TreeNode` là một **đối tượng** (*object*) với ba thuộc tính: `.val`, `.left` và `.right`.
Hai thuộc tính con lưu **tham chiếu** tới nút khác hoặc `None` khi không có con.
Gốc là nút duy nhất không có cha. **Lá** là nút không có cả con trái lẫn con phải.
Ví dụ gốc 8 có trái 3 và phải 10. Chúng ta đọc `root.left.val` để lấy 3,
không phải `root[1]` như truy cập list. LeetCode đã cấp sẵn cây cho lời giải;
chỉ trong notebook mới cần hàm `tree([8,3,10])` để dựng dữ liệu thử.
Đọc đoạn code và nói vì sao hai nút 3, 10 là hai đối tượng độc lập.

In [ ]:
example_root = TreeNode(8)
example_root.left = TreeNode(3)
example_root.right = TreeNode(10)
print('root:', example_root.val, 'left:', example_root.left.val, 'right:', example_root.right.val)
print('root.left is root.right?', example_root.left is example_root.right)
assert example_root.left.val == 3 and example_root.right.val == 10

### 2.2. `None`, `is None` và short-circuit tránh lỗi

Nếu `node is None`, biến đó **không trỏ đến nút thật**. Khi ấy `node.val`
sẽ gây `AttributeError`; một thuật toán cây phải kiểm tra tồn tại trước.
Biểu thức `node is not None and node.left is not None` dùng **short-circuit**:
vế sau chỉ được đánh giá nếu vế trước đúng. Đây là an toàn dữ liệu, không phải thủ thuật.
`is` kiểm tra cùng đối tượng, khác `==` thường dùng so sánh giá trị.
Với bài duyệt cây, hãy phân biệt *nút trống* với *nút có giá trị 0*:
`TreeNode(0)` là một nút thật, không được bỏ qua chỉ vì `.val == 0`.

In [ ]:
absent = None
real_zero = TreeNode(0)
for candidate in (absent, real_zero, TreeNode(-5)):
    exists = candidate is not None
    print('node=',candidate,'tồn tại?',exists,'val=',candidate.val if exists else 'không có')
assert real_zero is not None and absent is None

### 2.3. Đọc dữ liệu level-order và `None` trong ví dụ

LeetCode thường hiển thị cây theo *level-order*: các giá trị theo tầng, trái trước phải,
dùng `null` cho vị trí trống. Trong Python notebook ta viết `None`.
Chú ý list `[1, None, 2, 3]` **không** có nghĩa nút 3 là con phải của 2.
Khi dựng cây đúng kiểu level-order, gốc 1 có con phải 2; sau đó 2 có con trái 3.
Hãy dùng `show_tree(...)` để đọc ngược lại cây đã tạo. Đó là hàm học tập, **không nộp** LeetCode.
Đừng tự gán quan hệ cha-con theo công thức `2*i+1` khi có các ô `None` không đủ trong list.

In [ ]:
special = tree([1,None,2,3])
print('cấu trúc:',show_tree(special))
print('gốc:',special,'con phải:',special.right,'cháu bên trái:',special.right.left)
print('con trái gốc:',special.left)
assert special.right.left.val == 3 and special.left is None

### 2.4. Cú pháp `list` như stack LIFO

*Stack* (ngăn xếp) tuân theo **LIFO**: vào sau → ra trước.
Với Python, `stack.append(x)` là **đẩy** (*push*) và `stack.pop()` là **lấy/loại phần tử cuối**.
`pop()` vừa trả phần tử, vừa thay đổi chính list; khi rỗng thì gọi `pop()` sẽ gây lỗi.
Vì vậy chuẩn viết là `while stack:` rồi mới `stack.pop()`.
Trong duyệt cây, thứ tự `append` quyết định chính xác thứ tự xử lý;
mọi ví dụ nên in *stack trước pop*, *phần tử vừa pop* và *stack sau cập nhật*.

In [ ]:
plate_stack=[]
for label in ('đáy','giữa','đỉnh'):
    plate_stack.append(label)
print('stack lúc đầu:',plate_stack)
while plate_stack:
    print('pop ->',plate_stack.pop(),'còn:',plate_stack)
assert plate_stack == []

### 2.5. Vì sao phải đẩy con PHẢI trước con TRÁI?

Khi mục tiêu cần **xử lý nhánh trái trước nhánh phải**, quy tắc LIFO có vẻ ngược đời:
hãy đẩy **phải trước**, rồi đẩy **trái sau** để trái nằm trên đỉnh stack.
Đây là quy tắc thường gặp khi mô phỏng DFS bằng stack, nhưng mỗi kiểu traversal
còn quy định *thời điểm ghi giá trị gốc* khác nhau.
Ví dụ stack ban đầu `['root']`: pop root, append right, append left;
lần pop tiếp theo sẽ là left, không phải right. Dự đoán kết quả trước khi chạy.

In [ ]:
demo_stack=['root']
current=demo_stack.pop()
demo_stack.append('right')
demo_stack.append('left')
print('đã xử lý:',current,'stack còn:',demo_stack)
print('thứ tự các việc tiếp theo:',demo_stack.pop(),demo_stack.pop())
assert demo_stack == []

### 2.6. Đệ quy, base case, call stack và chiều cao cây

**Recursion (đệ quy)** là hàm gọi lại chính nó trên bài toán nhỏ hơn;
*base case* là điểm dừng (trong cây thường `node is None`).
Khi `dfs(root.left)` chưa trả về, lời gọi cha đang được giữ trong *call stack*.
Mỗi lần đi sâu thêm 1 tầng phải giữ thêm một lời gọi. Nếu `h` là số nút của
đường từ gốc đến lá dài nhất, đệ quy cần `O(h)` bộ nhớ ngăn xếp.
Cây lệch quá sâu có thể vượt **recursion limit** Python; giải pháp stack tường minh
cho phép viết DFS lặp và tránh giới hạn độ sâu gọi hàm.
Một `print` trong đệ quy chỉ minh họa; kết quả lời giải phải được trả bằng `return`.

In [ ]:
def enter_exit(n, indent=0):
    print('  '*indent+'vào', n)
    if n == 0:
        print('  '*indent+'dừng')
        return 0
    child=enter_exit(n-1,indent+1)
    print('  '*indent+'quay về',n)
    return 1+child
assert enter_exit(3)==3

### 2.7. Tham chiếu, đối tượng giống giá trị và output `list[int]`

Hai nút đều có `.val == 5` vẫn có thể là hai **nút khác nhau**;
traversal phải ghi **cả hai lần** số 5 nếu cây thật sự chứa hai nút như vậy.
`a = node` không tạo ra bản sao nút; cả hai tên cùng trỏ đối tượng.
Tuy nhiên `result.append(node.val)` ghi **giá trị số** vào list output, không ghi node.
Khi LeetCode yêu cầu `List[int]`, trả `[5,5]` là hợp lệ trong ví dụ có hai nút cùng giá trị.
Đừng tự dùng `set` khử trùng vì làm mất các lần xuất hiện có ý nghĩa.

In [ ]:
left_five=TreeNode(5)
right_five=TreeNode(5)
twin_root=TreeNode(1,left_five,right_five)
print('giá trị bằng?', left_five.val == right_five.val)
print('cùng đối tượng?', left_five is right_five)
print('số nút thật có giá trị 5:',sum(node.val==5 for node in (left_five,right_five)))
assert left_five is not right_five

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Stack giữ cặp `(node,visited)`: `visited=False` nghĩa chưa xử lý con; `True` nghĩa **con đã được sắp lịch trước** và bây giờ mới ghi node.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
stack=[('root',False)]
node,visited=stack.pop()
print('pop:',node,visited)
stack.extend([('root',True),('right',False),('left',False)])
print('thứ tự pop kế tiếp:',[stack.pop() for _ in range(3)])

### Câu hỏi gợi mở trước khi tối ưu

Sau khi tự chạy ví dụ nền tảng, hãy tự trả lời bằng lời:

1. Trong bài **Binary Tree Postorder Traversal**, đâu là dữ liệu thật, đâu là công cụ dựng thử?
2. Biến nào thay đổi sau một vòng lặp / sau một lời gọi đệ quy?
3. Khi nào một ký tự / node / giá trị được xem là 'đã xử lý xong'?
4. Nếu input rỗng, có một phần tử, hay chỉ có một nhánh, cần return kiểu gì?
5. Thao tác nào tăng theo số phần tử n, thao tác nào chỉ giữ vài biến?

Không cần trả lời bằng công thức ngay; thử input nhỏ rồi diễn đạt kết quả.

### Quy trình chạy notebook không phụ thuộc trạng thái ngầm

Các hàm dựng input, baseline, `solve`, trace và đáp án được định nghĩa theo thứ tự từ trên xuống dưới.
Không chạy phần đáp án trước phần bài tập nếu muốn tự luyện.
Khi thử một biến thể, hãy tạo input mới, không sửa input đã được dùng cho `assert` ở các cell phía trên.
Nếu chạy lại một cell, bạn nên kiểm tra các biến demo bị ghi đè; cách ổn định nhất là **Restart Kernel & Run All**.

Ở các bài Linked List/Tree, LeetCode đã cung cấp sẵn lớp node. Trong notebook, lớp node được định nghĩa để mô phỏng local và không phải mã cần nộp.

### 3.1. Ba thứ tự traversal nhìn trên cùng một cây

*Traversal* (duyệt cây) nghĩa là thăm các nút theo quy tắc cố định.
Trên cây có gốc 4, con trái 2 (với con 1,3), con phải 6 (với con 5,7):
**preorder** Root–Left–Right là `[4,2,1,3,6,5,7]`;
**inorder** Left–Root–Right là `[1,2,3,4,5,6,7]`;
**postorder** Left–Right–Root là `[1,3,2,5,7,6,4]`.
Bài #145 yêu cầu duy nhất **Left → Right → Root**. Đừng đổi thứ tự theo vị trí list input.
Có thể dùng cùng cấu trúc cây để nhìn sự khác biệt **thời điểm ghi `.val`**.
Ba thứ tự đều phải ghi đủ số nút thật, kể cả những nút trùng giá trị.

In [ ]:
chosen_tree=tree([4,2,6,1,3,5,7])
preorder_sample=[4,2,1,3,6,5,7]
inorder_sample=[1,2,3,4,5,6,7]
postorder_sample=[1,3,2,5,7,6,4]
print('cây:',show_tree(chosen_tree))
print('pre:',preorder_sample,'in:',inorder_sample,'post:',postorder_sample)
assert len(preorder_sample)==len(postorder_sample)==len(inorder_sample)==7

### 3.2. Vì sao DFS khác với duyệt level-order BFS?

**DFS (Depth-First Search)** đi sâu một nhánh trước khi quay sang nhánh khác;
preorder/postorder là các **thứ tự ghi kết quả khi DFS**.
**BFS (Breadth-First Search)** đi từng tầng, thường dùng queue FIFO.
Với cây `[1,2,3,4,5]`, BFS là `[1,2,3,4,5]`,
trong khi preorder và postorder thường khác hẳn.
Hiểu sự khác nhau này giúp tránh nhầm bài yêu cầu traversal bằng
thuật toán chỉ in theo hàng chờ. Nếu đề hỏi 'level order', mới dùng BFS.
`deque.popleft()` lấy phần tử đầu của queue, còn `list.pop()` lấy cuối của stack.

In [ ]:
from collections import deque
def level_order_demo(root):
    if root is None:return []
    q=deque([root]);values=[]
    while q:
        node=q.popleft();values.append(node.val)
        if node.left is not None:q.append(node.left)
        if node.right is not None:q.append(node.right)
    return values
small_tree=tree([1,2,3,4,5])
print('BFS:',level_order_demo(small_tree),'DFS bài đang học:',baseline(small_tree) if 'baseline' in globals() else '(sẽ học sau)')
assert level_order_demo(small_tree)==[1,2,3,4,5]

## 4. Cách nghĩ trực giác (baseline)

Đệ quy rất tự nhiên: thăm trái, thăm phải, rồi ghi gốc. Dễ nhớ nhưng hãy hiểu lý do: postorder cần biết cả hai cây con đã xong.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(root):
    result=[]
    def dfs(node):
        if node is None:return
        dfs(node.left);dfs(node.right);result.append(node.val)
    dfs(root);return result

In [ ]:
for a in [[],[1,None,2,3],[4,2,6,1,3,5,7]]:print(a,'->',baseline(tree(a)))

### 4.1. Đệ quy giúp diễn đạt thứ tự duyệt bằng chính câu tiếng Việt

Cách baseline dùng `dfs(node)` nhận một cây con:
nếu node rỗng thì không làm gì; nếu có node, gọi trên trái/phải
và ghi giá trị vào `result` đúng thời điểm yêu cầu.
Với #145, thứ tự phải là **Left → Right → Root**. Mỗi nút được ghi đúng một lần.
Tư duy recursion giúp người mới nhìn bài toán **cây con cùng dạng với cây lớn**.
Nhưng call stack có thể sâu O(h); code giải chính dùng stack tường minh
để tránh rủi ro `RecursionError` trên cây dạng dây rất sâu.
Hãy kiểm cùng ví dụ với một nút, một nhánh và cây cân đối.

In [ ]:
for arr in ([],[8],[8,3,None,2],[4,2,6,1,3,5,7]):
    root=tree(arr)
    print('cây:',arr,'baseline:',baseline(root))
    assert len(baseline(root))==sum(x is not None for x in arr)

### 4.2. Recursion trông gọn nhưng bộ nhớ ngăn xếp không miễn phí

Với cây một nhánh sâu h, `dfs` gọi thêm một lần cho mỗi node.
Các lời gọi cha phải chờ con trả về nên Python giữ tới O(h) stack frames.
Nếu h đủ lớn, Python có thể báo lỗi `maximum recursion depth exceeded`.
Điều này không làm thuật toán đệ quy sai, nhưng khiến nó kém bền vững
khi input vượt recursion limit của môi trường thực thi.
Dùng explicit stack đưa trạng thái vào `list`, vẫn có O(h) extra memory,
nhưng không phụ thuộc giới hạn call stack Python. Tránh tăng recursionlimit tùy tiện.

In [ ]:
def make_left_chain_demo(n):
    root=None
    for value in range(n-1,-1,-1):
        root=TreeNode(value,root,None)
    return root
chain=make_left_chain_demo(12)
print('cây dây n=12, thứ tự duyệt:',baseline(chain))
assert len(baseline(chain))==12

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Muốn làm bằng stack mà vẫn đúng thứ tự, ta lưu **trạng thái đã thăm**. Đặt `(node,True)` xuống dưới hai công việc con để ghi node sau cùng.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

### Bước đột phá — Khi cần ghi *sau* con, phải lưu trạng thái công việc

Đệ quy mặc định có thể `dfs(left); dfs(right); append(node.val)`.
Nhưng stack list chỉ lưu node sẽ không biết **khi nào hai cây con đã xong**.
Ta đặt thêm `done=False/True`: khi chưa xong, push một công việc ghi node **sau**
hai công việc giải cây con. Để tránh nhầm LIFO, push `(node,True)` trước,
`(right,False)` tiếp và `(left,False)` cuối. Khi pop `(node,True)`, mới append.

**Bất biến của stack:** mỗi công việc True nằm dưới tất cả công việc còn phải làm
cho các cây con của node ấy. Vì vậy khi True được pop, hai cây con đã hoàn tất,
và node được ghi đúng thứ tự Left–Right–Root.
Mỗi nút tạo đúng một cặp False/True ⇒ không mất, không ghi lặp.

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Pop `(node,done)`: nếu done, ghi giá trị; nếu chưa, push `(node,True)`, push `(right,False)`, rồi `(left,False)`. LIFO khiến trái xử lý trước phải và cả hai trước gốc.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Chuyển lời gọi đệ quy thành công việc nằm trên stack

Đệ quy giấu một ngăn xếp lời gọi; lời giải lặp **đặt ngăn xếp ra trước mắt**.
Với bài #145, muốn đạt thứ tự **Left → Right → Root**, ta cần lên kế hoạch chính xác
*khi nào ghi node* và *khi nào xử lý hai cây con*.
Việc thăm node không đồng nghĩa luôn phải ghi node ngay: preorder ghi sớm,
postorder ghi sau khi con hoàn thành.
**Bất biến** nên mô tả stack đang chứa những công việc nào chưa làm;
mỗi pop giải quyết một công việc hoặc tạo thêm các công việc con.
Để đúng, mọi nút thật phải được ghi đúng một lần và không bỏ sót con nào.

In [ ]:
task_stack=['root']
print('công việc ban đầu:',task_stack)
task=task_stack.pop()
print('đã lấy:',task)
task_stack.extend(['right','left'])
print('hai việc đang chờ:',task_stack,'vì pop cuối trước')
assert task_stack.pop()=='left' and task_stack.pop()=='right'

In [ ]:
def solve(root: Optional[TreeNode]) -> list[int]:
    if root is None:return []
    stack=[(root,False)];ans=[]
    while stack:
        node,done=stack.pop()
        if done:
            ans.append(node.val)
        else:
            stack.append((node,True))
            if node.right is not None:stack.append((node.right,False))
            if node.left is not None:stack.append((node.left,False))
    return ans

### 6.1. Postorder phải trì hoãn việc ghi node tới khi hoàn thành hai con

**Postorder** theo thứ tự Left → Right → Root.
Nếu ghi giá trị gốc ngay sau khi `pop()`, bạn tạo preorder, không phải postorder.
Lời giải dùng cặp `(node,done)` với `done=False` nghĩa là chưa xử lý hai cây con,
`done=True` nghĩa hai cây con đã được **lên lịch xử lý phía trên stack**.
Khi gặp `(node,False)`, push `(node,True)` trước, sau đó push con phải và con trái.
Vì LIFO, công việc con trái → con phải chạy trước công việc ghi node.
Đây là kỹ thuật mô phỏng lời gọi trở về sau đệ quy.

In [ ]:
root=tree([1,None,2,3])
stack=[(root,False)];result=[]
while stack:
    node,done=stack.pop()
    if done:result.append(node.val)
    else:
        stack.append((node,True))
        if node.right is not None:stack.append((node.right,False))
        if node.left is not None:stack.append((node.left,False))
print('postorder:',result)
assert result==[3,2,1]

### 6.2. Ngăn xếp `(node,True)` đứng DƯỚI công việc của hai con

Với node r có con trái l và con phải p, sau khi pop `(r,False)`, ta push:
thứ nhất `(r,True)`, thứ hai `(p,False)`, thứ ba `(l,False)`.
Từ đỉnh xuống, stack là `(l,False)`, `(p,False)`, `(r,True)`.
Do đó việc ghi r chỉ xảy ra **sau khi** giải quyết hết công việc trong hai cây con.
Khi r không có con, ta vẫn push `(r,True)` rồi pop và ghi r ở lượt kế tiếp.
Trạng thái boolean không có nghĩa node được ghi hai lần;
chỉ công việc `(r,True)` mới thật sự thêm vào result.

In [ ]:
pending=[]
pending.append(('root',True))
pending.append(('right',False))
pending.append(('left',False))
print('thứ tự công việc khi pop:',[pending.pop() for _ in range(3)])
assert pending == []

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Lượt | Công việc pop | Công việc mới | ans |
|---:|---|---|---|
| 1 | `(1,False)` | `(1,True)`, `(2,False)` | `[]` |
| 2 | `(2,False)` | `(2,True)`, `(3,False)` | `[]` |
| 3 | `(3,False)` | `(3,True)` | `[]` |
| 4 | `(3,True)` | ghi 3 | `[3]` |
| cuối | `(1,True)` | ghi 1 | `[3,2,1]` |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [ ]:
def trace_post(root):
    st=[(root,False)] if root else [];ans=[]
    while st:
        node,done=st.pop()
        print('pop',node.val,'đã-xong-con?',done)
        if done:ans.append(node.val);print('  ghi:',ans)
        else:
            st.append((node,True))
            if node.right:st.append((node.right,False))
            if node.left:st.append((node.left,False))
    return ans
trace_post(tree([1,None,2,3]))

### 7.1. Trace chi tiết postorder: trạng thái `(node,done)` trên stack

Trace phải hiện rõ cả hai lần gặp một node: `(node,False)` để lên lịch con,
`(node,True)` mới được ghi vào kết quả.
Với `[4,2,6,1,3,5,7]`, nút 4 xuất hiện hai công việc khác nhau,
nhưng số 4 chỉ xuất hiện **một** lần trong output cuối và đứng cuối.
Ta in danh sách cặp `(val,done)` trước/sau pop để chứng minh con được xử lý trước.
Khi một node là lá, nó vẫn phải trải qua hai lượt công việc trước khi ghi.

In [ ]:
def trace_postorder_full145(root):
    stack=[(root,False)] if root else [];out=[];turn=0
    while stack:
        turn+=1
        print('trước',turn,':',[(n.val,done) for n,done in stack])
        node,done=stack.pop()
        if done:out.append(node.val);print('  GHI',node.val,'=>',out)
        else:
            print('  lên lịch con của',node.val)
            stack.append((node,True))
            if node.right:stack.append((node.right,False))
            if node.left:stack.append((node.left,False))
    return out
assert trace_postorder_full145(tree([2,1,3]))==[1,3,2]

### 7.2. Postorder trên cây chỉ có một nhánh: đi xuống rồi quay lên

Với cây lệch phải 1→2→3→4, postorder là `[4,3,2,1]`:
đệ quy hoặc stack phải xử lý sâu nhất trước, sau đó mới quay lại các cha.
Đây là cách kiểm rất tốt để phân biệt postorder và preorder:
preorder ở cùng cây sẽ là `[1,2,3,4]`.
Cây rỗng trả list rỗng; cây một nút trả list chứa đúng một số.
Hãy đối chiếu baseline và solve để thấy cả hai đều đúng.
Đừng kết luận từ một cây có một nút vì cả 3 traversal đều giống nhau.

In [ ]:
for arr in ([],[1],[1,None,2,None,3,None,4],[1,2,None,3,None,4]):
    result=solve(tree(arr))
    print('cây:',arr,'postorder:',result)
    assert result==baseline(tree(arr))
assert solve(tree([1,None,2,None,3,None,4]))==[4,3,2,1]

### 7.3. Lời giải khác: Root–Right–Left rồi đảo lại output

Một cách iterative khác là duyệt `Root → Right → Left`,
lưu dãy này rồi đảo bằng `[::-1]` để được `Left → Right → Root`.
Cách này dễ nhớ nếu đã hiểu preorder, nhưng tạo thêm thao tác đảo output
và có thể tốn thêm `O(n)` bộ nhớ tạm cho lát cắt.
Cách `done` của lời giải chính biểu diễn trực tiếp postorder,
nên rất có giá trị sư phạm khi cần 'xử lý node sau con'.
Ta giữ phương án đảo làm **bản đối chiếu độc lập**, không coi nó là phép màu.

In [ ]:
def postorder_reverse145(root):
    if root is None:return []
    stack=[root];reversed_visit=[]
    while stack:
        node=stack.pop();reversed_visit.append(node.val)
        if node.left:stack.append(node.left)
        if node.right:stack.append(node.right)
    return reversed_visit[::-1]
for arr in ([],[1,None,2,3],[4,2,6,1,3,5,7]):
    assert postorder_reverse145(tree(arr))==solve(tree(arr))
print('PASS đối chiếu cách đảo với done flag')

### 7.4. Mỗi nút chỉ được GHI một lần, dù có hai công việc

Có cặp `(node,False)` và `(node,True)` không có nghĩa output nhân đôi.
Công việc False chỉ tạo các công việc con; **không append**.
Công việc True mới thêm node.val vào kết quả.
Mỗi node thật được tạo công việc False đúng một lần từ cha của nó,
và công việc True đúng một lần từ chính công việc False.
Vì vậy mỗi node được ghi **đúng một lần**.
Dùng dữ liệu có nhiều nút cùng val để không lẫn số lần ghi với số giá trị khác nhau.

In [ ]:
for arr in ([1,1,1],[5,5,5,5,5],[1,None,2,3]):
    values=solve(tree(arr))
    n_nodes=sum(x is not None for x in arr)
    print('cây:',arr,'=>',values,'số phần tử output:',len(values),'n:',n_nodes)
    assert len(values)==n_nodes and values==baseline(tree(arr))

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Với một node chưa làm, stack chỉ đưa nó vào kết quả sau khi cả nhánh trái và phải đã được xử lý, vì `(node,True)` nằm dưới các công việc của con. LIFO cũng đảm bảo trái được xét trước phải. Mỗi node ghi đúng một lần, nên thứ tự là Left-Right-Root.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
from typing import Optional
class Solution:
    def postorderTraversal(self, root: Optional[TreeNode]) -> list[int]:
        if root is None:
            return []
        stack = [(root, False)]
        result = []
        while stack:
            node, done = stack.pop()
            if done:
                result.append(node.val)
            else:
                stack.append((node, True))
                if node.right is not None:
                    stack.append((node.right, False))
                if node.left is not None:
                    stack.append((node.left, False))
        return result

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for a,expected in [([],[]),([1],[1]),([1,None,2,3],[3,2,1]),([4,2,6,1,3,5,7],[1,3,2,5,7,6,4])]:
    assert solve(tree(a))==expected
    assert Solution().postorderTraversal(tree(a))==expected
    assert baseline(tree(a))==expected
print('PASS 4 ca postorder')

### 10.1. Kiểm thử cây nhỏ sinh ngẫu nhiên rồi đối chiếu recursion

Một bộ test mạnh có cấu trúc cây đa dạng hơn 3 ví dụ trong đề.
Ta sinh cây kích thước 0..35 nút; mỗi node có thể được nối trái hoặc phải.
Đồng thời có thể có **giá trị trùng** để phát hiện lỗi vô tình khử trùng.
Trên cây nhỏ, baseline recursion an toàn và đóng vai trò **oracle độc lập**:
`solve(root)` phải trùng kết quả `baseline(root)`.
Hãy sử dụng `random.Random(seed)` để kết quả tái hiện được khi có lỗi.
Bài kiểm thử không sửa cây thật và chỉ in số ca PASS, tránh output nghìn dòng.

In [ ]:
import random
rng=random.Random(144 if 'preorder' in solve.__name__ else 145)
def random_tree_for_test(n,rng):
    if n==0:return None
    nodes=[TreeNode(rng.randrange(-3,4)) for _ in range(n)]
    possible=[nodes[0]]
    for child in nodes[1:]:
        while True:
            parent=rng.choice(possible)
            options=[side for side in ('left','right') if getattr(parent,side) is None]
            if not options:
                possible.remove(parent)
                continue
            side=rng.choice(options);setattr(parent,side,child)
            possible.append(child)
            if parent.left is not None and parent.right is not None:possible.remove(parent)
            break
    return nodes[0]
for n in range(36):
    for _ in range(12):
        root=random_tree_for_test(n,rng)
        assert solve(root)==baseline(root)
print('PASS 432 cây ngẫu nhiên kích thước 0..35')

### 10.2. Cây dây rất sâu để kiểm iterative stack

Cách recursion có thể bị giới hạn bởi call stack Python;
trong khi explicit stack có thể xử lý cây dây vài nghìn nút.
Ta dựng cây chỉ có con trái với 2.000 node và chạy hàm `solve`.
**Không chạy baseline recursion ở trường hợp này** để tránh RecursionError,
chỉ kiểm output gồm đủ 2.000 phần tử và đúng hai đầu được suy ra bằng tay.
Test này chỉ nhằm cho thấy ưu điểm của giải pháp iterative khi h lớn;
memory explicit stack vẫn O(h), không phải O(1).

In [ ]:
chain_root=make_left_chain_demo(2000)
long_output=solve(chain_root)
assert len(long_output)==2000
print('cây dây 2000 node: số phần tử:',len(long_output),
      '5 phần tử đầu:',long_output[:5], '5 phần tử cuối:',long_output[-5:])

### 10.3. Kiểm thử thuộc tính output: đủ số node và giữ nguyên đa tập giá trị

Dù chọn preorder hay postorder, thuật toán phải ghi mỗi node đúng một lần;
vì vậy `len(output)` bằng số node thực, và `Counter(output)`
phải bằng tần suất `.val` của toàn bộ nút.
Kiểm thuộc tính này độc lập với thứ tự, sau đó tiếp tục dùng baseline
để xác nhận chính xác **thứ tự**. Một output có đủ số nhưng đảo thứ tự
vẫn sai đề, do đó cần cả hai lớp kiểm tra.
Nhấn mạnh thuật toán traversal **không khử trùng giá trị**.
Dùng vài cây có node cùng số, số âm và node rỗng.

In [ ]:
from collections import Counter
trees=[[],[5],[5,5,5],[1,2,3,None,4,2,2],[0,-1,1,-1,None,0,0]]
for values in trees:
    out=solve(tree(values))
    raw=[x for x in values if x is not None]
    assert Counter(out)==Counter(raw)
    assert len(out)==len(raw)
    assert out==baseline(tree(values))
print('PASS bảo toàn số node và đa tập giá trị')

## 11. Time / Space Complexity — có tính tới stack và output

Gọi `n` là số node thật và `h` là chiều cao cây. Duyệt **postorder** cần đi tới mỗi node đúng một lần: Time **O(n)**.
Hàm đệ quy baseline giữ call stack tới O(h); lời giải lặp giữ explicit stack tới O(h) công việc tại thời điểm cao nhất (tối đa O(n) nếu cây hình dây).
Danh sách **output luôn chứa n giá trị** nên riêng bộ nhớ trả kết quả cần O(n), tách khỏi extra working memory O(h).
Đối với cây đầy, h có thể O(log n); với cây lệch, h có thể O(n). Không được tuyên bố mọi cây đều cao O(log n).
Giải pháp iterative tránh RecursionError do giới hạn gọi hàm Python; nó không làm mất chi phí bộ nhớ lưu công việc.

## 12. Các lỗi người mới hay mắc

- ❌ Ghi node ngay khi pop lần đầu → preorder.
- ❌ Push trái trước phải trong LIFO → thứ tự con bị ngược.
- ❌ Không phân biệt `done` → lặp lại node hoặc ghi sai.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

In [ ]:
root_bug145=tree([1,2,3])
print('cây có hai con:',show_tree(root_bug145),'đúng postorder:',solve(root_bug145))
assert solve(root_bug145)==[2,3,1]

### 12.1. Sai lầm: append ngay ở công việc `(node,False)`

Nếu thêm node.val ngay khi vừa pop `(node,False)`, gốc đi vào output trước con.
Đó chính là preorder, trong khi bài #145 yêu cầu postorder.
Cặp `(node,True)` không phải để ghi lần hai;
chỉ **giai đoạn True** được quyền append.
Test `[1,2,3]` sẽ cho sai khác rõ: bản sai `[1,2,3]`, bản đúng `[2,3,1]`.
Nhờ có test nhỏ này, ta nhận ra lỗi do **thời điểm ghi**,
không phải do giá trị node hay việc kiểm tra None.

In [ ]:
def wrong_early_write145(root):
    if root is None:return []
    stack=[root];out=[]
    while stack:
        node=stack.pop();out.append(node.val)
        if node.right:stack.append(node.right)
        if node.left:stack.append(node.left)
    return out
test_root=tree([1,2,3])
print('viết sớm:',wrong_early_write145(test_root),'đúng:',solve(test_root))
assert wrong_early_write145(test_root)!=solve(test_root)

### 12.2. Sai lầm: đảo thứ tự push hai con trong postorder

Dù node chỉ được ghi ở trạng thái True, thứ tự push vẫn quan trọng.
Nếu push con trái trước con phải thì LIFO xử lý **phải** trước **trái**,
output sẽ thành Right–Left–Root, không phải Left–Right–Root.
Chỉ một cây có cả hai con `[1,2,3]` đủ phân biệt:
bản sai `[3,2,1]`, đúng `[2,3,1]`.
Vì vậy test một nhánh không đủ chứng minh đúng.
Quy tắc ổn định: push `(node,True)`, rồi `(right,False)`, rồi `(left,False)`.

In [ ]:
def wrong_child_order145(root):
    if root is None:return []
    stack=[(root,False)];out=[]
    while stack:
        node,done=stack.pop()
        if done:out.append(node.val)
        else:
            stack.append((node,True))
            if node.left:stack.append((node.left,False))
            if node.right:stack.append((node.right,False))
    return out
print('sai:',wrong_child_order145(tree([1,2,3])),'đúng:',solve(tree([1,2,3])))
assert wrong_child_order145(tree([1,2,3]))==[3,2,1]

## 13. Nhận diện dạng bài và vận dụng

Cần xử lý node **sau khi tất cả phần phụ thuộc hoàn tất** → postorder. Stack có nhãn trạng thái giúp mô phỏng đệ quy.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tự vẽ ba thứ tự preorder, inorder, postorder cho cùng một cây; chỉ khác lúc ghi node nhưng cả ba thăm tất cả node.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
print('Postorder cây cân đối:',solve(tree([4,2,6,1,3,5,7])))

### 14.1. Biến thể: postorder rất hợp để tổng hợp kết quả từ cây con

Trong nhiều bài cây, thông tin của node cha chỉ tính được khi đã biết
kết quả của hai cây con. Ví dụ số node trong cây con của node cha bằng
`1 + số nút bên trái + số nút bên phải`.
Đây là lý do postorder thường đi cùng **Dynamic Programming trên cây**.
Ta viết ví dụ đệ quy đếm số node: trước tiên giải trái, giải phải rồi cộng cho cha.
Với cây rỗng return 0, với cây một nút return 1.
Biến thể giúp bạn hiểu *khi nào cần giải con trước*, không chỉ cách in list.

In [ ]:
def subtree_size145(node):
    if node is None:return 0
    left_count=subtree_size145(node.left)
    right_count=subtree_size145(node.right)
    return 1+left_count+right_count
for arr in ([],[1],[1,2,3,None,4]):
    print(arr,'số nút:',subtree_size145(tree(arr)))
    assert subtree_size145(tree(arr))==len([x for x in arr if x is not None])

### 14.2. Biến thể: tìm chiều cao cây bằng postorder

Chiều cao của node bằng `1 + max(height(left), height(right))`.
Để biết chiều cao node cha, phải tính chiều cao hai cây con trước:
đây chính là mô hình postorder (Left–Right–Root).
Với cây rỗng, height 0; cây một nút, height 1.
Với cây `[1,2,3,4,None,None,5]`, chiều cao là 3.
Dù công thức ngắn, hãy giải thích rõ **chứng minh từ cây con**:
đường dài nhất đến lá phải đi qua một trong hai con,
nên lấy max của hai chiều cao rồi cộng 1 cho nút hiện tại.

In [ ]:
def tree_height145(node):
    if node is None:return 0
    left_h=tree_height145(node.left)
    right_h=tree_height145(node.right)
    return 1+max(left_h,right_h)
for arr,expected in [([],0),([8],1),([1,2,3,4,None,None,5],3)]:
    print(arr,'height:',tree_height145(tree(arr)))
    assert tree_height145(tree(arr))==expected

## 15. Hệ thống luyện tập — 6 bài từ cơ bản đến nâng cao

**Bài 1 — Dự đoán output Python:** Stack gồm `(root,True)`, `(right,False)`, `(left,False)`; dự đoán thứ tự ba pop liên tiếp.

**Bài 2 — Trace tay:** Trace `[1,None,2,3]`: ghi các cặp `(node,done)` và chỉ append `.val` khi done=True.

**Bài 3 — Hoàn thiện code chính:** tự viết hàm `exercise_postorder145(root)` mà không nhìn cell nộp.

**Bài 4 — Biến thể:** Viết hàm tính CHIỀU CAO cây bằng postorder: `height(None)=0`, `height(node)=1+max(height(left),height(right))`.

**Bài 5 — Viết test phản ví dụ:** Tự tạo 5 test `(level_order,expected_postorder)` gồm cây rỗng, cây một nút, cây dây, cây cân đối và node trùng giá trị.

**Bài 6 — Mini challenge:** Biến thể: kiểm tra cây có cân bằng chiều cao hay không (chênh lệch height của hai con tại MỌI node không quá 1).

**Cách tự học:** làm bài 1–2 trên giấy → viết TODO bài 3–5 → tự kiểm bằng test của chính mình → thử bài 6. Hãy ghi rõ lý do chọn trạng thái, ranh giới vòng lặp và điều kiện dừng; **đáp án tham khảo nằm sau toàn bộ đề bài**.

### Bài 3 — TODO: tự viết lại Binary Tree Postorder Traversal

Viết code vào hàm ở cell tiếp theo theo đúng kiểu input/output của đề.
Không gọi hàm TODO khi chạy toàn bộ notebook. Khi hoàn thành, tự bỏ `raise NotImplementedError` rồi chạy test được gợi ý dưới hàm.
**Tiêu chí tự chấm:** giải thích được vì sao code đúng ngay cả ở trường hợp nhỏ nhất, và không làm thao tác dư thừa so với lời giải chính.

In [ ]:
def exercise_postorder145(root):
    # TODO 1: nếu root None return []
    # TODO 2: stack chứa (node,done=False)
    # TODO 3: khi done=False, push (node,True), rồi right, rồi left
    # TODO 4: khi done=True, append node.val
    raise NotImplementedError("Tự giải trước khi chạy test")
# assert exercise_postorder145(tree([1,None,2,3]))==[3,2,1]

### Bài 4 — TODO: đổi yêu cầu, giữ kỹ thuật cốt lõi

Viết hàm tính CHIỀU CAO cây bằng postorder: `height(None)=0`, `height(node)=1+max(height(left),height(right))`.

Đừng chép y nguyên code submit: hãy chỉ ra đầu ra thay đổi ở đâu; tự tạo 2 input mới để kiểm thử.

In [ ]:
def exercise_height_postorder145(root):
    # TODO: base case None -> 0; tính hai chiều cao con rồi 1+max.
    raise NotImplementedError("Chưa làm bài 4")
# assert exercise_height_postorder145(tree([1,2,3,4])) == 3

### Bài 5 — TODO: tạo bộ kiểm thử có chủ đích

Tự tạo 5 test `(level_order,expected_postorder)` gồm cây rỗng, cây một nút, cây dây, cây cân đối và node trùng giá trị.

Mỗi test phải có **expected tự tính trước**, không được lấy `expected=solve(input)`. Một test tốt thường có mục tiêu bắt một kiểu sai cụ thể. Nếu có node, dựng dữ liệu mới ở mỗi lượt test.

In [ ]:
def exercise_cases145():
    # TODO: trả [(level_order_list, expected_postorder_list), ...] ít nhất 5 ca.
    raise NotImplementedError("Chưa làm bài 5")

### Bài 6 — Mini challenge chuyển giao kỹ thuật (khó)

**Yêu cầu:** Biến thể: kiểm tra cây có cân bằng chiều cao hay không (chênh lệch height của hai con tại MỌI node không quá 1).

    Bài 6 không thuộc đề gốc. Hãy tự phân tích **miền input, output, tính đúng**
    và chọn một phản ví dụ nhỏ trước khi code. Nếu thay đổi cấu trúc lời giải,
    cần nói rõ điều kiện nào đã thay đổi so với bài gốc.
    Không gọi hàm có TODO trong lần chạy toàn bộ notebook: để nguyên `NotImplementedError`
    cho tới khi bạn tự điền mã và tự bật test.
    **Kiểm tra:** so kết quả với đáp án cuối, nhưng chỉ sau khi đã tự làm.

In [ ]:
def exercise_balanced145(root):
    # TODO: duyệt postorder, mỗi cây con trả chiều cao hoặc dấu hiệu mất cân bằng.
    # Mọi node đều phải thỏa abs(height(left)-height(right)) <= 1.
    raise NotImplementedError("Mini challenge chưa giải")
# assert exercise_balanced145(tree([1,2,3])) is True

## 16. Gợi ý theo ba cấp — chỉ đọc sau khi tự thử

**Cấp 1:** postorder khác preorder ở **thời điểm ghi nút**: sau cả hai con.
**Cấp 2:** trạng thái `(node,True)` là công việc ghi node được đặt ở dưới các việc con trong stack.
**Cấp 3:** khi node chưa xong, push `(node,True)`, `(right,False)`, `(left,False)` đúng thứ tự ấy.
**Nâng cao:** bài 6 dùng postorder trả `-1` nếu cây con đã mất cân bằng, nếu chưa thì trả chiều cao; tại mỗi node kiểm chênh lệch hai con.

**Cách sử dụng gợi ý:** bắt đầu ở cấp 1; nếu vẫn chưa ra, chỉ mở cấp 2 rồi tự viết tiếp. Không đọc lời giải đầy đủ ngay khi vừa gặp khó. Với bài 6, thử tự xác định input nhỏ nhất gây sai trong phương án bạn đề xuất.

## 17. Đáp án tham khảo — chỉ đọc sau khi đã làm TODO

Các hàm **đáp án** dùng tên khác với `exercise_*` để không vô tình thực thi code chưa hoàn thiện.
Đáp án bài 3 được định nghĩa trước, rồi đến các biến thể bài 4–6 và bộ test tương ứng.
Sau khi chạy, hãy so sánh logic từng dòng với lời giải chính và tìm lý do các cài đặt tương đương, không chỉ so con số output.

In [ ]:
def exercise_core_answer(root):
    if root is None:return []
    stack=[(root,False)];ans=[]
    while stack:
        node,done=stack.pop()
        if done:
            ans.append(node.val)
        else:
            stack.append((node,True))
            if node.right is not None:stack.append((node.right,False))
            if node.left is not None:stack.append((node.left,False))
    return ans

print("Đã định nghĩa đáp án tham khảo; xem lại từng dòng rồi tự gõ lại.")

### Đáp án bài 4 — biến thể với yêu cầu đầu ra mới

Bài 4 yêu cầu: Viết hàm tính CHIỀU CAO cây bằng postorder: `height(None)=0`, `height(node)=1+max(height(left),height(right))`.

    So với lời giải gốc, hãy chỉ ra phần nào giữ nguyên:
    cú pháp duyệt, trạng thái trước/sau, bất biến. Phần thay đổi phải tương ứng
    đúng với **đầu ra mới**, không làm thay chữ ký code submit #
    hoặc vô tình lẫn thêm điều kiện ngoài phạm vi bài gốc.
    Đoạn đáp án dưới đây có assert riêng để kiểm tính đúng.

In [ ]:
def answer_height_postorder145(root):
    if root is None:return 0
    return 1+max(answer_height_postorder145(root.left),answer_height_postorder145(root.right))
for values,expected in [([],0),([1],1),([1,2,3],2),([1,2,3,4],3)]:
    assert answer_height_postorder145(tree(values))==expected
print('PASS đáp án bài 4 — tính chiều cao')

### Đáp án bài 6 và kiểm thử đáp án bài 5

**Bài 6:** Biến thể: kiểm tra cây có cân bằng chiều cao hay không (chênh lệch height của hai con tại MỌI node không quá 1).

    Đọc suy luận trước khi so code: giữ trạng thái nào; tại mỗi bước
    trạng thái cập nhật ra sao; vì sao khi dừng, kết quả đáp ứng yêu cầu.
    Bài 5 là tự xây bộ test: đáp án tham khảo còn có các mẫu độc lập,
    chạy kèm assert nhưng **không hề chạy TODO của người học**.
    Nếu bộ test bạn tự nghĩ khác, điều đó bình thường: hãy đảm bảo cả
    test của bạn và test tham khảo đều có expected đúng.

In [ ]:
def answer_balanced145(root):
    def height_or_minus_one(node):
        if node is None:return 0
        left=height_or_minus_one(node.left)
        if left==-1:return -1
        right=height_or_minus_one(node.right)
        if right==-1 or abs(left-right)>1:return -1
        return 1+max(left,right)
    return height_or_minus_one(root)!=-1
for values,expected in [([],True),([1],True),([1,2,3],True),([1,2,None,3,None,4],False),([3,9,20,None,None,15,7],True)]:
    assert answer_balanced145(tree(values))==expected
print('PASS đáp án mini challenge cây cân bằng')

cases145=[([],[]),([1],[1]),([1,2,3],[2,3,1]),([1,None,2,3],[3,2,1]),([5,5,5],[5,5,5]),([4,2,6,1,3,5,7],[1,3,2,5,7,6,4])]
for values,expected in cases145:assert solve(tree(values))==expected
print('PASS 6 test tham khảo postorder')

### Lời giải thích và đáp án ngắn cho cả 6 bài

**Bài 1:** pop lần lượt `(left,False)`, `(right,False)`, `(root,True)`.
**Bài 2:** `[1,None,2,3]` cho `[3,2,1]`: ghi 3 trước 2, cuối cùng mới 1.
**Bài 3:** `exercise_core_answer` dùng cờ done để trì hoãn ghi node đến sau hai con.
**Bài 4:** `answer_height_postorder145` trả `1+max` hai chiều cao con.
**Bài 5:** bộ test còn kiểm cây có cả hai nhánh và node trùng để tránh nhầm traversal.
**Bài 6:** `answer_balanced145` trả -1 ngay khi một cây con mất cân bằng, không cần thăm tiếp.

Đối với bài tập viết code, hãy so sánh cả **điều kiện lặp**, **thứ tự cập nhật** và **cách return** chứ không chỉ so giá trị cuối. Các hàm `answer_*` của bài 4 và 6 nằm ở những cell ngay phía trên để người học có thể chạy riêng sau khi tự làm.

## 18. Tóm tắt và checklist trước khi nộp

**Tư duy phải tự diễn đạt được bằng lời:** nêu bài toán bằng một câu, chỉ rõ trạng thái nào đủ để quyết định bước kế tiếp, rồi mới viết code. Đọc lại các ví dụ khó trong notebook; khi một điều kiện `if` làm bạn phân vân, hãy lập bảng giá trị **ngay trước và sau** câu lệnh ấy.

1. Tôi đã phân biệt đầu vào/đầu ra, và đọc đúng chữ ký `class Solution` của bài.
2. Tôi đã tự chạy thử ví dụ nhỏ, biên, phản ví dụ và kiểm thử đối chiếu nhiều cách giải.
3. Tôi hiểu từng cấu trúc Python, có thể giải thích vì sao dùng nó thay vì cách khác.
4. Tôi chỉ ra được bất biến: sau bước thứ k, điều gì **luôn đúng** và tại sao?
5. Tôi phân tích Time và Space dựa trên số lượng phần tử/nút/bít và có tính tới output/stack/chuỗi tạm.
6. Tôi đã hoàn thành TODO **trước** khi kéo tới đáp án tham khảo.
7. Tôi chỉ sao chép `class Solution` để nộp LeetCode; các hàm dựng input, trace và bộ test dành cho notebook, không dành cho submit.

**Tự đánh giá:** bạn hiểu bài khi tự viết được lời giải trên một trang trống và xử lý được một biến thể mà không sao chép mã cũ.

**Mẫu nhận dạng của bài 145:** DFS phải xử lý Left/Right trước Root → stack có cờ `done`, append khi done=True.